# Annotating tools for an AI: a reference you can run

When your function becomes an **MCP tool**, the AI never sees your code. It reads exactly
three things:

1. the **name** of the function,
2. the **description**: your docstring, word for word,
3. the **input schema**: a JSON description of every parameter, built from your **type hints**.

The schema is not just a hint: the MCP server **checks every call against it** and rejects bad
inputs before your code even runs. So good annotations both *tell* the AI what to send and
*stop* it from sending nonsense.

**How to use this notebook:** run the cells from top to bottom (Shift+Enter). Every example
prints what the AI would receive. Part 9 has exercises with automatic checks.

**Open it with the project's conda environment**, or the imports below will fail:
`conda activate tool-jumper`, then `jupyter lab learn_annotations.ipynb`. In VS Code, choose the
**tool-jumper** kernel (top right).

## 0. Setup

The cell below imports everything this notebook uses and defines three small helpers:

* `show(function)`: print what the AI reads about a function;
* `try_call(function, inputs)`: call it **through MCP**, the way the AI does;
* `check(function, wanted)`: used by the exercises in part 9.

You do not need to understand the helpers to use them.

In [ ]:
import json
from typing import Annotated, Any, Literal

from mcp.server import MCPServer
from pydantic import Field


async def read_like_the_ai(tool_function):
    """Register one function on a fresh MCP server and return the tool the AI would receive."""
    lesson_server = MCPServer("annotation-lesson")
    lesson_server.tool()(tool_function)
    listed_tools = await lesson_server.list_tools()
    first_tool = listed_tools[0]
    return first_tool


async def show(tool_function):
    """Print what the AI reads about a function: its name, description and input schema."""
    tool = await read_like_the_ai(tool_function)
    print(f"NAME:        {tool.name}")
    print(f"DESCRIPTION: {tool.description}")
    print("INPUT SCHEMA:")
    print(json.dumps(tool.input_schema, indent=2))


async def try_call(tool_function, inputs):
    """Call a function THROUGH MCP, like the AI does, and print whether it was accepted."""
    lesson_server = MCPServer("annotation-lesson")
    lesson_server.tool()(tool_function)
    try:
        result = await lesson_server.call_tool(tool_function.__name__, inputs)
        print(f"ACCEPTED {inputs} -> {result.content[0].text}")
    except Exception as error:
        # Keep the lines that explain the problem; drop the web links.
        reasons = [line for line in str(error).splitlines() if "errors.pydantic.dev" not in line]
        print(f"REJECTED {inputs}:")
        print("\n".join(reasons[:7]))


async def check(tool_function, wanted):
    """Compare a function's schema with what an exercise asks for; print PASS or FAIL."""
    tool = await read_like_the_ai(tool_function)
    properties = tool.input_schema.get("properties", {})
    passed_count = 0
    total_count = 0
    for input_name, wanted_rules in wanted.items():
        input_schema = properties.get(input_name, {})
        for rule_name, wanted_value in wanted_rules.items():
            actual_value = input_schema.get(rule_name)
            # For "description" any text counts; every other rule must match exactly.
            is_passed = (
                bool(actual_value) if rule_name == "description" else (actual_value == wanted_value)
            )
            shown_value = "some text" if rule_name == "description" else wanted_value
            print(
                f"{'PASS' if is_passed else 'FAIL'}  {input_name}: {rule_name} should be "
                f"{shown_value!r}, got {actual_value!r}"
            )
            passed_count += int(is_passed)
            total_count += 1
    word_count = len((tool.description or "").split())
    has_real_docstring = word_count >= 8
    print(
        f"{'PASS' if has_real_docstring else 'FAIL'}  docstring has at least 8 words "
        f"(has {word_count})"
    )
    passed_count += int(has_real_docstring)
    total_count += 1
    print(f"\n{passed_count}/{total_count} passed")

## 1. Imports cheat sheet: two imports cover everything

**`typing` is enough for most inputs.** It gives the type, choices, optional inputs and lists:

| You want | Write | Import |
|---|---|---|
| a plain type | `name: str`, `steps: int`, `speed: float`, `is_on: bool` | none |
| an optional input | `note: str \| None = None`, `steps: int = 10` | none |
| a list | `names: list[str]` | none |
| a fixed set of choices | `direction: Literal["left", "right"]` | `from typing import Literal` |

**`Field` from pydantic adds rules and per-input descriptions**, which `typing` cannot express:

| You want | Write | Import |
|---|---|---|
| a range | `steps: Annotated[int, Field(ge=0, le=200)]` | `from typing import Annotated` and `from pydantic import Field` |
| a text length | `note: Annotated[str, Field(min_length=1, max_length=100)]` | same |
| a sentence about one input | `Annotated[int, Field(description="Pixels to walk.")]` | same |

`Annotated[TYPE, Field(...)]` means: "this input is a TYPE, **and** here are extra rules".
`pydantic` is already installed: the MCP library itself uses it to build every schema.
You do not need `enum`: `Literal` is the simpler way to list choices, and small models read it best.

## 2. Plain types

`str`, `int`, `float` and `bool` become `string`, `integer`, `number` and `boolean`.
Every input without a default is **required**.

In [ ]:
def set_speed(robot_name: str, steps: int, scale: float, is_running: bool) -> str:
    """Change how a robot moves."""
    return "ok"


await show(set_speed)

### No type hint? It becomes "text"

Without a type hint, the AI is told the input is a **string**, even when its default is a
number. Here `b=3` is described as text: the AI may well send `"3"` or `"three"`.

In [ ]:
def mystery(a, b=3):
    """Do something."""
    return "ok"


await show(mystery)

## 3. Defaults and optional inputs

An input with a default is **not required**, and the default appears in the schema.
`str | None = None` means "text, or nothing at all".

In [ ]:
def remember(note: str | None = None, importance: int = 1) -> str:
    """Save a note for later."""
    return "saved"


await show(remember)

## 4. Choices: `Literal`

`Literal[...]` lists the only values allowed. In the schema it becomes an **enum**, the
clearest possible instruction: the AI sees the exact words it may use.

In [ ]:
def turn(direction: Literal["left", "right"]) -> str:
    """Turn the robot to face left or right."""
    return f"facing {direction}"


await show(turn)

And it is enforced: anything else is rejected **before** `turn` runs.

In [ ]:
await try_call(turn, {"direction": "left"})
await try_call(turn, {"direction": "up"})

## 5. Number rules: `Annotated` + `Field`

| `Field(...)` | Schema | Meaning |
|---|---|---|
| `ge=0` / `le=200` | `minimum` / `maximum` | at least 0 / at most 200 |
| `gt=0` / `lt=10` | `exclusiveMinimum` / `exclusiveMaximum` | more than 0 / less than 10 |
| `multiple_of=5` | `multipleOf` | 0, 5, 10, 15, ... |

In [ ]:
def walk_steps(
    steps: Annotated[int, Field(ge=0, le=200)],
    speed: Annotated[float, Field(gt=0, lt=10)],
) -> str:
    """Walk a number of steps at a speed."""
    return f"walked {steps} steps"


await show(walk_steps)

In [ ]:
await try_call(walk_steps, {"steps": 50, "speed": 2.5})
await try_call(walk_steps, {"steps": 500, "speed": 0})  # both rules broken: two reasons
await try_call(walk_steps, {"steps": "15", "speed": 1})  # text "15" is quietly turned into 15

### Why not just `typing`?

You might try to describe a rule with plain `typing`, by putting a sentence or a number in
`Annotated`. The MCP server **ignores** it: the sentence is not in the schema, and nothing is
enforced. Here the AI sends 5000 and the call is accepted.

In [ ]:
def walk_far(steps: Annotated[int, "How many steps, 0 to 200."]) -> str:
    """Walk a number of steps."""
    return f"walked {steps} steps"


await show(walk_far)
await try_call(walk_far, {"steps": 5000})

That is what `Field` is for: the same rule, written so the schema carries it and the server enforces it.

In [ ]:
def walk_limited(
    steps: Annotated[int, Field(ge=0, le=200, description="How many steps; one step is 5 px.")],
) -> str:
    """Walk a number of steps."""
    return f"walked {steps} steps"


await show(walk_limited)
await try_call(walk_limited, {"steps": 5000})

## 6. Text rules

| `Field(...)` | Schema | Meaning |
|---|---|---|
| `min_length=1` | `minLength` | not empty |
| `max_length=100` | `maxLength` | at most 100 characters |
| `pattern=r"^[a-z ]+$"` | `pattern` | must match this regular expression |

In [ ]:
def name_robot(
    name: Annotated[str, Field(min_length=1, max_length=20, pattern=r"^[A-Za-z]+$")],
) -> str:
    """Give the robot a one-word name made of letters."""
    return f"Hello, {name}!"


await show(name_robot)
await try_call(name_robot, {"name": "Beep"})
await try_call(name_robot, {"name": "R2-D2"})

## 7. Describe each input: `description` and `examples`

`Field(description=...)` puts a sentence **inside the schema, next to that input**. That is
where the AI looks when it fills in the value. `examples=[...]` shows typical values.

In [ ]:
def walk(
    direction: Literal["left", "right"],
    distance: Annotated[
        int,
        Field(ge=0, le=1000, description="How far to walk, in pixels.", examples=[50, 120]),
    ],
) -> str:
    """Walk the robot left or right by a distance in pixels (0 to 1000)."""
    return f"walked {distance} px {direction}"


await show(walk)

### Lists

`list[str]` is a list of strings. `Field(max_length=3)` on a list limits **how many items**
it may have (`maxItems` in the schema).

In [ ]:
def remember_many(notes: Annotated[list[str], Field(max_length=3)]) -> str:
    """Save up to 3 short notes at once."""
    return f"saved {len(notes)} notes"


await show(remember_many)

## 8. Docstrings and return values

**The whole docstring becomes the description**, word for word. A Google-style `Args:`
section is **not** turned into per-input descriptions: it stays plain text in the description.
To attach a sentence to one input, use `Field(description=...)` (part 7).

In [ ]:
def jump(direction: Literal["left", "right"]) -> str:
    """Jump once toward the left or the right (about 165 px sideways).

    Only works while the robot stands on the ground.

    Args:
        direction: which way to jump.

    """
    return f"jumped {direction}"


await show(jump)

**Return a short string.** `-> str` tells MCP the tool answers with text. Make that text a
**fact the AI can act on**, in the units it acts in:

| Bad answer | Good answer |
|---|---|
| `{"x": 310, "terrain": [...]}` (raw numbers to decode) | `"A gap 100 px wide is 260 px (52 steps) ahead."` |
| `"JUMP NOW!!!"` (an order) | `"You are 0 px from the gap's edge; it is 100 px wide."` |
| `"Error"` | `"Not moving: the robot is broken (fell_into_gap). Restart the stage first."` |

### What NOT to do

Each of these confuses the AI. Run the cell and look at the schemas.

* `Any` gives the AI **no type at all**.
* `*args` and `**kwargs` become two strange **required** text inputs called `args` and `kwargs`.
* Vague names (`d`, `n`, `x`) tell the AI nothing.

In [ ]:
def vague(d: Any, *args, **kwargs) -> str:
    """Do it."""
    return "?"


await show(vague)

## 9. Exercises

Edit each function so it passes its check. Re-run the function's cell, then its check cell.
Do not change the function names.

### Exercise 1: only two directions
`hop` should accept only `"left"` or `"right"`, and have a docstring of at least 8 words
saying what it does.

In [ ]:
def hop(direction):
    """Hop."""
    return f"hopped {direction}"

In [ ]:
await check(hop, {"direction": {"enum": ["left", "right"]}})

### Exercise 2: a note with limits
`save_note` should take text that is **1 to 100 characters long**, with a **description** of
what makes a good note, and a docstring of at least 8 words.

In [ ]:
def save_note(note):
    """Save."""
    return f"saved: {note}"

In [ ]:
await check(
    save_note,
    {"note": {"type": "string", "minLength": 1, "maxLength": 100, "description": True}},
)

### Exercise 3: walking in steps
`walk_steps_safely` takes a `direction` (`"left"` or `"right"`) and `steps`: a whole number
from **0 to 200**, with a **description** that says one step is 5 pixels. Docstring of at
least 8 words.

In [ ]:
def walk_steps_safely(direction, steps):
    """Walk."""
    return f"walked {steps} steps {direction}"

In [ ]:
await check(
    walk_steps_safely,
    {
        "direction": {"enum": ["left", "right"]},
        "steps": {"type": "integer", "minimum": 0, "maximum": 200, "description": True},
    },
)

## 10. Checklist for every tool in `team_tools/`

Before you hand in, go through each tool:

- [ ] The **name** says what it does (`walk_to_next_edge`, not `go`).
- [ ] The **docstring** says what it does, when to use it, what it returns, and when it refuses.
- [ ] Every input has a **type hint**; no `Any`, no `*args`/`**kwargs`.
- [ ] Inputs with a fixed set of values use **`Literal`**.
- [ ] Numbers have **ranges** (`ge`/`le`), text has **lengths** (`min_length`/`max_length`).
- [ ] Inputs whose meaning is not obvious have **`Field(description=...)`**, with units.
- [ ] It returns **one short string** (at most 300 characters) of **facts**, with units.
- [ ] It **never crashes**: robot errors are caught and explained in the returned text.

Then check what the AI really reads about your team's tools:

```bash
python check_tools.py --schema
```